# Klingon Translator - Fine-Tuning NLLB-200

Fine-tunes Facebook's NLLB-200 (distilled 600M) for English <-> Klingon translation.

**Setup:** Runtime > Change runtime type > **A100 GPU** (recommended) or T4 GPU

**Training data:** ~22,600 English-Klingon pairs from Tatoeba, OPUS, boQwI' dictionary, paq'batlh, and curated proverbs.

**Strategy:**
- Bidirectional training (en->tlh AND tlh->en)
- SentencePiece tokenizer extension for Klingon with byte fallback and apostrophe preservation
- Auto-scaled vocabulary (1000-4000 tokens based on corpus size)
- Transfer learning: initialize Klingon embeddings via base-tokenizer decomposition
- Pre-tokenized dataset for maximum GPU utilization
- GPU-adaptive config: auto-detects A100 vs T4 and optimizes accordingly
- 15 epochs with warmup and weight decay

## 1. Setup & GPU Check

In [ ]:
!pip install -q transformers sentencepiece datasets sacrebleu accelerate pyyaml

from google.colab import drive
drive.mount("/content/drive")

import json
import random
import shutil
import time
from pathlib import Path

import numpy as np
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    NllbTokenizerFast,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
import sentencepiece as spm

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_mem = 0
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")

    # Enable TF32 for A100+ (3x faster float32 matmuls via TensorCores)
    if gpu_mem >= 40:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        print("  TF32 enabled (A100 TensorCores)")
else:
    print("WARNING: No GPU detected! Training will be very slow.")
    print("Go to Runtime > Change runtime type > A100 GPU")

IS_A100 = gpu_mem >= 40
print(f"Mode: {'A100 (high-performance)' if IS_A100 else 'T4 (memory-conservative)'}")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Constants
BASE_MODEL = "facebook/nllb-200-distilled-600M"
KLINGON_CODE = "tlh_Latn"
ENGLISH_CODE = "eng_Latn"
PROJECT_DIR = Path("/content/drive/MyDrive/Klingon Translator")

# ── Copy data to local SSD for faster I/O ────────────────────────
# Google Drive FUSE is slow (~1-2 MB/s). Copying to /content/ (local SSD)
# eliminates network overhead for all data reads during training.
LOCAL_DIR = Path("/content/local_project")
if LOCAL_DIR.exists():
    shutil.rmtree(LOCAL_DIR)

print("\nCopying data to local SSD...")
t0 = time.time()
shutil.copytree(PROJECT_DIR / "data", LOCAL_DIR / "data")
print(f"  Copied in {time.time() - t0:.1f}s")

DATA_DIR = LOCAL_DIR / "data" / "processed"
RAW_DIR = LOCAL_DIR / "data" / "raw"

# Checkpoints save locally (fast), bulk-upload to Drive after training
CKPT_DIR = Path("/content/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Final model saves to Drive (one-time write after training)
MODELS_DIR = PROJECT_DIR / "models"
FINAL_DIR = MODELS_DIR / "fine-tuned"

print(f"\nData dir (local): {DATA_DIR}")
print(f"Checkpoints (local): {CKPT_DIR}")
print(f"Final model (Drive): {FINAL_DIR}")

## 2. Load Data

In [ ]:
def load_jsonl(path):
    """Load a JSONL file into a list of dicts."""
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_data = load_jsonl(DATA_DIR / "train.jsonl")
val_data = load_jsonl(DATA_DIR / "val.jsonl")
test_data = load_jsonl(DATA_DIR / "test.jsonl")

print(f"Train: {len(train_data):,} pairs")
print(f"Val:   {len(val_data):,} pairs")
print(f"Test:  {len(test_data):,} pairs")
print(f"Total: {len(train_data) + len(val_data) + len(test_data):,} pairs")
print()

# Preview some examples
for i, pair in enumerate(train_data[:5]):
    print(f"  [{i}] en: {pair['en']}")
    print(f"       tlh: {pair['tlh']}")
    print()

## 3. Extend Tokenizer with Klingon

In [ ]:
# Load base model and tokenizer
print(f"Loading base model: {BASE_MODEL}")
tokenizer = NllbTokenizerFast.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
model = model.to(device)
print(f"Base vocab size: {len(tokenizer):,}")
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

# Collect all Klingon text for SentencePiece training
# Include processed JSONL splits + raw sources (OPUS Moses, paq'batlh)
all_klingon = [p["tlh"] for p in train_data + val_data + test_data]

# Also include OPUS Moses Klingon file (raw parallel text)
opus_tlh = RAW_DIR / "opus" / "tatoeba" / "Tatoeba.en-tlh.tlh"
if opus_tlh.exists():
    opus_lines = [l.strip() for l in opus_tlh.read_text(encoding="utf-8").splitlines() if l.strip()]
    all_klingon.extend(opus_lines)
    print(f"Added {len(opus_lines):,} OPUS Moses lines")

# Also include paq'batlh pairs
paqbatlh_file = RAW_DIR / "paqbatlh_pairs.json"
if paqbatlh_file.exists():
    paq_pairs = json.loads(paqbatlh_file.read_text(encoding="utf-8"))
    paq_lines = [p["tlh"].strip() for p in paq_pairs if p.get("tlh", "").strip()]
    all_klingon.extend(paq_lines)
    print(f"Added {len(paq_lines):,} paq'batlh lines")

# Deduplicate
all_klingon = list(dict.fromkeys(all_klingon))  # preserve order, remove dupes

klingon_corpus = "/content/klingon_corpus.txt"
with open(klingon_corpus, "w", encoding="utf-8") as f:
    f.write("\n".join(all_klingon))
print(f"\nKlingon corpus: {len(all_klingon):,} unique sentences")

# Auto-scale vocab size based on corpus size (1000-4000)
num_unique = len(all_klingon)
auto_vocab_size = min(4000, max(1000, num_unique // 5))
print(f"Auto vocab_size: {auto_vocab_size} (from {num_unique} unique sentences)")

# Train SentencePiece BPE model on Klingon text
print(f"Training SentencePiece (BPE, vocab={auto_vocab_size})...")
spm.SentencePieceTrainer.train(
    input=klingon_corpus,
    model_prefix="/content/klingon_spm",
    vocab_size=auto_vocab_size,
    character_coverage=1.0,
    model_type="bpe",
    pad_id=3,
    hard_vocab_limit=False,
    byte_fallback=True,       # Unknown chars -> bytes instead of <unk>
    split_digits=True,         # Digits handled individually
    user_defined_symbols=["'"],  # Apostrophe is atomic (central to Klingon)
)

# Load trained SPM and find new tokens
klingon_sp = spm.SentencePieceProcessor()
klingon_sp.load("/content/klingon_spm.model")
print(f"Klingon SPM vocab: {klingon_sp.get_piece_size()} tokens")

# Test tokenization
for phrase in ["nuqneH", "tlhIngan maH", "Qapla'"]:
    pieces = klingon_sp.encode(phrase, out_type=str)
    print(f"  {phrase} -> {pieces}")

In [ ]:
# Find new tokens not in NLLB vocabulary
existing_vocab = set(tokenizer.get_vocab().keys())
new_tokens = []
for i in range(klingon_sp.get_piece_size()):
    token = klingon_sp.id_to_piece(i)
    if token.startswith("<") and token.endswith(">"):
        continue  # Skip control tokens
    if token not in existing_vocab:
        new_tokens.append(token)

print(f"New Klingon tokens to add: {len(new_tokens)}")

# ── Pre-compute embeddings BEFORE modifying tokenizer ────────────
# For each new token, tokenize with the ORIGINAL NLLB tokenizer and
# average those subword embeddings. This gives a semantically meaningful
# initialization instead of the global mean.
old_emb_size = model.get_input_embeddings().weight.shape[0]
token_init_map = {}  # token_string -> (input_emb_vec, output_emb_vec)

with torch.no_grad():
    input_emb = model.get_input_embeddings().weight
    output_emb = model.get_output_embeddings().weight
    mean_input = input_emb[:old_emb_size].mean(dim=0)
    mean_output = output_emb[:old_emb_size].mean(dim=0)

    for token in new_tokens:
        base_ids = tokenizer.encode(token, add_special_tokens=False)
        if base_ids:
            avg_in = input_emb[base_ids].mean(dim=0)
            avg_out = output_emb[base_ids].mean(dim=0)
        else:
            avg_in = mean_input
            avg_out = mean_output
        token_init_map[token] = (avg_in.clone(), avg_out.clone())

print(f"Pre-computed embeddings for {len(token_init_map)} new tokens")

# ── Now modify the tokenizer ─────────────────────────────────────

# Register Klingon language code
special_map = tokenizer.special_tokens_map
current_special = special_map.get("additional_special_tokens", [])
tokenizer.add_special_tokens(
    {"additional_special_tokens": current_special + [KLINGON_CODE]}
)
print(f"Registered language code: {KLINGON_CODE}")

# Add new subword tokens
num_added = tokenizer.add_tokens(new_tokens)
print(f"Added {num_added} tokens to tokenizer")

# Resize model embeddings
model.resize_token_embeddings(len(tokenizer))
new_emb_size = model.get_input_embeddings().weight.shape[0]
print(f"Resized embeddings: {old_emb_size:,} -> {new_emb_size:,}")

# ── Initialize new embeddings via transfer learning ──────────────
eng_id = tokenizer.convert_tokens_to_ids(ENGLISH_CODE)
tlh_id = tokenizer.convert_tokens_to_ids(KLINGON_CODE)

with torch.no_grad():
    input_emb = model.get_input_embeddings().weight
    output_emb = model.get_output_embeddings().weight

    for i in range(old_emb_size, new_emb_size):
        if i == tlh_id:
            # Klingon lang code <- English lang code
            input_emb[i] = input_emb[eng_id].clone()
            output_emb[i] = output_emb[eng_id].clone()
        else:
            # Look up pre-computed per-token embedding
            token_str = tokenizer.convert_ids_to_tokens(i)
            if token_str in token_init_map:
                init_in, init_out = token_init_map[token_str]
                input_emb[i] = init_in
                output_emb[i] = init_out
            else:
                # Fallback: global mean
                input_emb[i] = mean_input.clone()
                output_emb[i] = mean_output.clone()

print(f"\nInitialized {new_emb_size - old_emb_size} new embeddings")
print(f"  tlh_Latn (id={tlh_id}) <- eng_Latn (id={eng_id})")
print(f"  Other tokens <- per-token base-tokenizer decomposition")
print(f"\nFinal vocab size: {len(tokenizer):,}")

In [ ]:
# ── Tokenizer Quality Report ──────────────────────────────────────

print("=" * 60)
print("Tokenizer Quality Report")
print("=" * 60)

# Fertility: average subword tokens per whitespace word
total_words = 0
total_tokens = 0
for sent in all_klingon:
    words = sent.split()
    total_words += len(words)
    token_ids = tokenizer.encode(sent, add_special_tokens=False)
    total_tokens += len(token_ids)

fertility = total_tokens / total_words if total_words > 0 else 0
print(f"\nFertility: {fertility:.2f} tokens/word")
print(f"  ({total_tokens:,} tokens for {total_words:,} words across {len(all_klingon):,} sentences)")

# Coverage: how many Klingon SPM tokens were new vs already in NLLB
klingon_spm_vocab = sum(
    1 for i in range(klingon_sp.get_piece_size())
    if not (klingon_sp.id_to_piece(i).startswith("<") and klingon_sp.id_to_piece(i).endswith(">"))
)
already_in_nllb = klingon_spm_vocab - num_added
coverage_pct = (num_added / klingon_spm_vocab) * 100 if klingon_spm_vocab > 0 else 0

print(f"\nVocab coverage:")
print(f"  Klingon SPM vocab: {klingon_spm_vocab}")
print(f"  New tokens added:  {num_added} ({coverage_pct:.1f}%)")
print(f"  Already in NLLB:   {already_in_nllb} ({100 - coverage_pct:.1f}%)")

# Sample tokenizations
sample_phrases = [
    "Qapla'",
    "tlhIngan maH",
    "nuqneH",
    "bortaS bIr jablu'DI' reH QaQqu' nay'",
    "Heghlu'meH QaQ jajvam",
    "DabuQlu'DI' yISuv",
]
print("\nSample tokenizations:")
for phrase in sample_phrases:
    ids = tokenizer.encode(phrase, add_special_tokens=False)
    tokens = tokenizer.convert_ids_to_tokens(ids)
    print(f"  {phrase}")
    print(f"    -> {tokens} ({len(tokens)} tokens)")

print("=" * 60)

## 4. Create Bidirectional Training Dataset

In [ ]:
from torch.utils.data import Dataset

class BilingualDataset(Dataset):
    """Bidirectional translation dataset with pre-tokenization.

    For each pair, creates two training examples:
    - English -> Klingon
    - Klingon -> English

    All tokenization happens once in __init__() and results are cached
    in memory. This eliminates the CPU tokenization bottleneck during
    training — __getitem__() is a simple list lookup.
    """

    def __init__(self, pairs, tokenizer, max_length=128):
        self.cached = []  # Pre-tokenized examples

        print(f"  Pre-tokenizing {len(pairs):,} pairs (2 directions each)...")
        t0 = time.time()

        for p in pairs:
            # en -> tlh
            tokenizer.src_lang = ENGLISH_CODE
            src = tokenizer(
                p["en"], truncation=True, max_length=max_length
            )
            tokenizer.src_lang = KLINGON_CODE
            tgt = tokenizer(
                p["tlh"], truncation=True, max_length=max_length
            )
            tgt_lang_id = tokenizer.convert_tokens_to_ids(KLINGON_CODE)
            self.cached.append({
                "input_ids": src["input_ids"],
                "attention_mask": src["attention_mask"],
                "labels": [tgt_lang_id] + tgt["input_ids"],
            })

            # tlh -> en
            tokenizer.src_lang = KLINGON_CODE
            src = tokenizer(
                p["tlh"], truncation=True, max_length=max_length
            )
            tokenizer.src_lang = ENGLISH_CODE
            tgt = tokenizer(
                p["en"], truncation=True, max_length=max_length
            )
            eng_lang_id = tokenizer.convert_tokens_to_ids(ENGLISH_CODE)
            self.cached.append({
                "input_ids": src["input_ids"],
                "attention_mask": src["attention_mask"],
                "labels": [eng_lang_id] + tgt["input_ids"],
            })

        # Shuffle once (deterministic with global seed)
        random.shuffle(self.cached)

        elapsed = time.time() - t0
        print(f"  Done: {len(self.cached):,} examples in {elapsed:.1f}s")

    def __len__(self):
        return len(self.cached)

    def __getitem__(self, idx):
        return self.cached[idx]

train_dataset = BilingualDataset(train_data, tokenizer)
val_dataset = BilingualDataset(val_data, tokenizer)

print(f"\nTraining examples: {len(train_dataset):,} (2x {len(train_data):,} pairs)")
print(f"Validation examples: {len(val_dataset):,} (2x {len(val_data):,} pairs)")

# Preview an example
ex = train_dataset[0]
print(f"Sample input_ids length: {len(ex['input_ids'])}")
print(f"Sample labels length: {len(ex['labels'])}")

## 5. Fine-Tune

Training configuration adapts to GPU:

**A100 (≥40GB VRAM):**
- **Batch size 32** (no gradient accumulation needed)
- **BF16** mixed precision (native A100, more stable than FP16)
- **TF32** enabled for float32 matmuls (TensorCore acceleration)
- **No gradient checkpointing** (trade memory for speed)
- **4 dataloader workers** with pinned memory
- Checkpoints save to local SSD (fast), bulk-upload to Drive after training

**T4 fallback (16GB VRAM):**
- **Batch size 4** with **gradient accumulation 8** (effective batch = 32)
- **FP16** mixed precision
- **Gradient checkpointing** enabled to fit in memory

**Common:** 15 epochs, learning rate 2e-5 with warmup, best model by val loss

In [ ]:
# Free memory before training
import gc
torch.cuda.empty_cache()
gc.collect()

# ── GPU-adaptive training configuration ──────────────────────────
if IS_A100:
    # A100: maximize throughput — large batch, bf16, no grad checkpointing
    batch_size = 32
    grad_accum = 1
    use_bf16 = True
    use_fp16 = False
    num_workers = 4
    pin_memory = True
    print("A100 config: batch=32, bf16, no gradient checkpointing")
else:
    # T4: conserve memory — small batch, fp16, gradient checkpointing
    batch_size = 4
    grad_accum = 8
    use_bf16 = False
    use_fp16 = True
    num_workers = 2
    pin_memory = False
    model.gradient_checkpointing_enable()
    print("T4 config: batch=4, fp16, gradient checkpointing enabled")

effective_batch = batch_size * grad_accum

training_args = Seq2SeqTrainingArguments(
    output_dir=str(CKPT_DIR),  # Local SSD (fast saves)
    num_train_epochs=15,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=200,
    bf16=use_bf16,
    fp16=use_fp16,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    predict_with_generate=False,
    logging_steps=50,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=num_workers,
    dataloader_pin_memory=pin_memory,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, padding=True, label_pad_token_id=-100
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

print(f"\nStarting training...")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {batch_size} (x{grad_accum} accum = {effective_batch} effective)")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Precision: {'BF16' if use_bf16 else 'FP16'}")
print(f"  Workers: {num_workers}, pin_memory: {pin_memory}")
print(f"  Checkpoints: {CKPT_DIR} (local SSD)")
print(f"  GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated")
print()

train_result = trainer.train()

# Print training summary
print(f"\nTraining complete!")
print(f"  Total steps: {train_result.global_step}")
print(f"  Training loss: {train_result.training_loss:.4f}")
print(f"  Runtime: {train_result.metrics['train_runtime']:.0f}s")

## 6. Save Fine-Tuned Model

In [ ]:
# Save fine-tuned model to Google Drive (one-time upload)
FINAL_DIR.mkdir(parents=True, exist_ok=True)
print("Saving fine-tuned model to Google Drive...")
t0 = time.time()
model.save_pretrained(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))
print(f"  Saved to {FINAL_DIR} in {time.time() - t0:.0f}s")

# Also save to a local copy for faster loading
model.save_pretrained("/content/fine-tuned")
tokenizer.save_pretrained("/content/fine-tuned")
print("  Also saved local copy to /content/fine-tuned")

# Upload best checkpoint to Drive (optional backup)
DRIVE_CKPT_DIR = MODELS_DIR / "checkpoints"
if CKPT_DIR.exists() and any(CKPT_DIR.iterdir()):
    print(f"\nUploading checkpoints to Drive...")
    t0 = time.time()
    if DRIVE_CKPT_DIR.exists():
        shutil.rmtree(DRIVE_CKPT_DIR)
    shutil.copytree(str(CKPT_DIR), str(DRIVE_CKPT_DIR))
    print(f"  Uploaded to {DRIVE_CKPT_DIR} in {time.time() - t0:.0f}s")

## 7. Evaluate on Test Set

Compute BLEU scores for both translation directions on the held-out test set.

In [ ]:
import sacrebleu

def translate_batch(texts, src_lang, tgt_lang, batch_size=32):
    """Translate a list of texts in batches."""
    model.eval()
    results = []
    tgt_id = tokenizer.convert_tokens_to_ids(tgt_lang)

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        tokenizer.src_lang = src_lang
        inputs = tokenizer(
            batch, return_tensors="pt", padding=True, truncation=True,
            max_length=128
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=tgt_id,
                max_length=128,
                num_beams=5,
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        results.extend(decoded)

        if (i // batch_size) % 5 == 0:
            print(f"  Translated {min(i + batch_size, len(texts))}/{len(texts)}...")

    return results

# English -> Klingon
print("Evaluating English -> Klingon...")
en_texts = [p["en"] for p in test_data]
tlh_refs = [p["tlh"] for p in test_data]
tlh_preds = translate_batch(en_texts, ENGLISH_CODE, KLINGON_CODE)

bleu_en2tlh = sacrebleu.corpus_bleu(tlh_preds, [tlh_refs])
print(f"  BLEU (en->tlh): {bleu_en2tlh.score:.2f}")

# Klingon -> English
print("\nEvaluating Klingon -> English...")
tlh_texts = [p["tlh"] for p in test_data]
en_refs = [p["en"] for p in test_data]
en_preds = translate_batch(tlh_texts, KLINGON_CODE, ENGLISH_CODE)

bleu_tlh2en = sacrebleu.corpus_bleu(en_preds, [en_refs])
print(f"  BLEU (tlh->en): {bleu_tlh2en.score:.2f}")

print(f"\n{'='*50}")
print(f"RESULTS:")
print(f"  English -> Klingon BLEU: {bleu_en2tlh.score:.2f}")
print(f"  Klingon -> English BLEU: {bleu_tlh2en.score:.2f}")
print(f"  Average BLEU: {(bleu_en2tlh.score + bleu_tlh2en.score) / 2:.2f}")
print(f"{'='*50}")

## 8. Sample Translations

In [ ]:
# Test with some well-known phrases
test_phrases_en = [
    "Today is a good day to die.",
    "Success!",
    "What do you want?",
    "We are Klingons!",
    "Don't be silly!",
    "Where is the bathroom?",
    "I don't understand.",
    "Revenge is a dish best served cold.",
    "Only a fool fights in a burning house.",
]

test_phrases_tlh = [
    "Qapla'!",
    "nuqneH.",
    "tlhIngan maH!",
    "HIja'.",
    "ghobe'.",
    "yIDoghQo'!",
    "qatlho'.",
    "maj.",
]

print("=== English -> Klingon ===")
for phrase in test_phrases_en:
    tokenizer.src_lang = ENGLISH_CODE
    inputs = tokenizer(phrase, return_tensors="pt").to(device)
    tlh_id = tokenizer.convert_tokens_to_ids(KLINGON_CODE)
    with torch.no_grad():
        out = model.generate(**inputs, forced_bos_token_id=tlh_id, max_length=128, num_beams=5)
    result = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"  {phrase}")
    print(f"    -> {result}")
    print()

print("\n=== Klingon -> English ===")
for phrase in test_phrases_tlh:
    tokenizer.src_lang = KLINGON_CODE
    inputs = tokenizer(phrase, return_tensors="pt").to(device)
    eng_id = tokenizer.convert_tokens_to_ids(ENGLISH_CODE)
    with torch.no_grad():
        out = model.generate(**inputs, forced_bos_token_id=eng_id, max_length=128, num_beams=5)
    result = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"  {phrase}")
    print(f"    -> {result}")
    print()

## 9. Training Summary

After training completes, the fine-tuned model is saved to Google Drive at:
```
Klingon Translator/models/fine-tuned/
```

To use locally:
1. Download the `models/fine-tuned/` directory from Google Drive
2. Load with: `KlingonTranslator("models/fine-tuned")`

Performance notes:
- **A100 (80GB):** ~2-3 hours for 15 epochs (bf16, batch 32, local checkpoints)
- **T4 (16GB):** ~8-10 hours for 15 epochs (fp16, batch 4, gradient checkpointing)
- Data is copied to local SSD at start to avoid Drive I/O bottlenecks
- Dataset is pre-tokenized once — GPU never waits on CPU tokenization

Next steps:
- Try more epochs if BLEU is low
- Experiment with learning rate and batch size
- Add back-translation for data augmentation
- Build Gradio web UI (Phase 6)